# Multi-Seed Leakage Stress Test

This notebook strengthens the first leakage-stress result by repeating the core experiment across five random seeds.

Scope:

- Models: XGBoost and LightGBM
- Seeds: 42, 101, 202, 303, 404
- D1/D2 scenarios: strict safe pipeline vs SMOTE-before-split
- D3 scenarios: strict pre-transaction features vs ledger-state features

The notebook is resumable. Each seed/dataset/scenario/model result is saved to Drive as soon as it finishes.

In [ ]:
!pip -q install -U imbalanced-learn lightgbm xgboost seaborn pyarrow

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from pathlib import Path

# Drive screenshot showed: My Drive > Credit__ > Datasets
PROJECT_ROOT = Path('/content/drive/MyDrive/Credit__')

DATASET_SEARCH_DIRS = [
    PROJECT_ROOT / 'Datasets',
    PROJECT_ROOT / '00_source_datasets',
    PROJECT_ROOT,
]

RUN_NAME = 'leakage_stress_multiseed_v1'
EXPERIMENT_ROOT = PROJECT_ROOT / '03_leakage_stress_testing' / RUN_NAME
CHECKPOINT_DIR = EXPERIMENT_ROOT / 'checkpoints'
RESULTS_DIR = EXPERIMENT_ROOT / 'results'
FIGURES_DIR = EXPERIMENT_ROOT / 'figures'
LOGS_DIR = EXPERIMENT_ROOT / 'logs'

for d in [EXPERIMENT_ROOT, CHECKPOINT_DIR, RESULTS_DIR, FIGURES_DIR, LOGS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

SEEDS = [42, 101, 202, 303, 404]
TEST_SIZE = 0.20
VALIDATION_SIZE = 0.20

CAPS = {
    'D1_creditcard': None,
    'D2_fraudTest': 300_000,
    'D3_PaySim': 200_000,
}

SMOKE_TEST = False
SMOKE_CAP = 30_000

MODEL_NAMES = ['xgboost', 'lightgbm']

print('Project root:', PROJECT_ROOT)
print('Experiment root:', EXPERIMENT_ROOT)
print('Seeds:', SEEDS)

In [ ]:
import os
import json
import time
import warnings
from datetime import datetime

import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    average_precision_score,
    precision_score,
    recall_score,
    f1_score,
    matthews_corrcoef,
    roc_auc_score,
    confusion_matrix,
    brier_score_loss,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, RobustScaler

from imblearn.over_sampling import SMOTE

import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings('ignore')

try:
    import xgboost as xgb
except Exception as exc:
    xgb = None
    print('XGBoost unavailable:', exc)

try:
    import lightgbm as lgb
except Exception as exc:
    lgb = None
    print('LightGBM unavailable:', exc)

sns.set_theme(style='whitegrid', context='paper')
plt.rcParams.update({
    'figure.dpi': 150,
    'savefig.dpi': 600,
    'font.family': 'DejaVu Sans',
    'font.size': 9,
    'axes.titlesize': 10,
    'axes.labelsize': 9,
    'xtick.labelsize': 8,
    'ytick.labelsize': 8,
    'legend.fontsize': 8,
    'legend.title_fontsize': 8,
    'pdf.fonttype': 42,
    'ps.fonttype': 42,
})

def now():
    return datetime.now().strftime('%Y-%m-%d %H:%M:%S')

def write_json(path, obj):
    path = Path(path)
    tmp = path.with_suffix(path.suffix + '.tmp')
    tmp.write_text(json.dumps(obj, indent=2, default=str), encoding='utf-8')
    tmp.replace(path)

def write_csv_atomic(path, df):
    path = Path(path)
    tmp = path.with_suffix(path.suffix + '.tmp')
    df.to_csv(tmp, index=False)
    tmp.replace(path)

def append_log(message):
    line = f'[{now()}] {message}'
    print(line)
    with open(LOGS_DIR / 'run.log', 'a', encoding='utf-8') as f:
        f.write(line + '\n')

def checkpoint_path(seed, dataset, scenario, model_name):
    return CHECKPOINT_DIR / f'seed{seed}__{dataset}__{scenario}__{model_name}.csv'

def is_done(seed, dataset, scenario, model_name):
    return checkpoint_path(seed, dataset, scenario, model_name).exists()

In [ ]:
def find_dataset_file(file_name):
    candidates = []
    for directory in DATASET_SEARCH_DIRS:
        candidates.append(directory / file_name)
        if directory.exists():
            candidates.extend(directory.glob(f'**/{file_name}'))
    for path in candidates:
        if path.exists():
            return path
    raise FileNotFoundError(f'Could not find {file_name}')

DATASET_FILES = {
    'D1_creditcard': find_dataset_file('creditcard.csv'),
    'D2_fraudTest': find_dataset_file('fraudTest.csv'),
    'D3_PaySim': find_dataset_file('PS.csv'),
}

for name, path in DATASET_FILES.items():
    append_log(f'{name}: {path}')

write_json(EXPERIMENT_ROOT / 'dataset_file_manifest.json', {k: str(v) for k, v in DATASET_FILES.items()})

In [ ]:
def cap_by_retaining_fraud(df, label_col, cap, seed):
    if cap is None or len(df) <= cap:
        return df.sample(frac=1.0, random_state=seed).reset_index(drop=True)
    y = pd.to_numeric(df[label_col], errors='coerce')
    fraud = df[y == 1]
    legit = df[y == 0]
    legit_n = max(cap - len(fraud), 0)
    legit_sample = legit.sample(n=min(legit_n, len(legit)), random_state=seed)
    out = pd.concat([fraud, legit_sample], axis=0)
    return out.sample(frac=1.0, random_state=seed).reset_index(drop=True)

def load_dataset(dataset_name, seed):
    path = DATASET_FILES[dataset_name]
    df = pd.read_csv(path, low_memory=False)

    if dataset_name == 'D1_creditcard':
        df = df.drop_duplicates().reset_index(drop=True)
        label_col = 'Class'
    elif dataset_name == 'D2_fraudTest':
        label_col = 'is_fraud'
    elif dataset_name == 'D3_PaySim':
        label_col = 'isFraud'
        df = df.dropna(subset=[label_col]).reset_index(drop=True)
        df[label_col] = df[label_col].astype(int)
    else:
        raise ValueError(dataset_name)

    cap = SMOKE_CAP if SMOKE_TEST else CAPS[dataset_name]
    df = cap_by_retaining_fraud(df, label_col, cap, seed)

    fraud = int((pd.to_numeric(df[label_col], errors='coerce') == 1).sum())
    append_log(f'{dataset_name} seed={seed}: rows={len(df):,}, fraud={fraud:,}, rate={fraud/len(df)*100:.4f}%')
    return df, label_col

def prepare_features(df, dataset_name, feature_policy):
    df = df.copy()

    if dataset_name == 'D1_creditcard':
        y = df['Class'].astype(int)
        X = df[[f'V{i}' for i in range(1, 29)] + ['Amount']].copy()
        return X, y

    if dataset_name == 'D2_fraudTest':
        df['trans_dt'] = pd.to_datetime(df['trans_date_trans_time'], errors='coerce')
        df['dob_dt'] = pd.to_datetime(df['dob'], errors='coerce')
        df['trans_hour'] = df['trans_dt'].dt.hour
        df['trans_dayofweek'] = df['trans_dt'].dt.dayofweek
        df['trans_month'] = df['trans_dt'].dt.month
        df['age_years'] = (df['trans_dt'] - df['dob_dt']).dt.days / 365.25
        keep = ['category', 'amt', 'state', 'city_pop', 'trans_hour', 'trans_dayofweek', 'trans_month', 'age_years']
        return df[keep].copy(), df['is_fraud'].astype(int)

    if dataset_name == 'D3_PaySim':
        keep = ['type', 'amount', 'oldbalanceOrg', 'oldbalanceDest']
        if feature_policy == 'ledger_state':
            keep += ['newbalanceOrig', 'newbalanceDest']
        return df[keep].copy(), df['isFraud'].astype(int)

    raise ValueError(dataset_name)

def make_preprocessor(X):
    numeric_cols = X.select_dtypes(include=['number', 'bool']).columns.tolist()
    categorical_cols = [c for c in X.columns if c not in numeric_cols]
    numeric_pipe = Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', RobustScaler(with_centering=False)),
    ])
    categorical_pipe = Pipeline([
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('onehot', OneHotEncoder(handle_unknown='ignore', min_frequency=50, sparse_output=True)),
    ])
    return ColumnTransformer([
        ('num', numeric_pipe, numeric_cols),
        ('cat', categorical_pipe, categorical_cols),
    ])

In [ ]:
def build_model(model_name, seed, scale_pos_weight=1.0):
    if model_name == 'xgboost':
        if xgb is None:
            raise RuntimeError('XGBoost not available')
        return xgb.XGBClassifier(
            n_estimators=300,
            max_depth=6,
            learning_rate=0.05,
            subsample=0.85,
            colsample_bytree=0.85,
            objective='binary:logistic',
            eval_metric='aucpr',
            tree_method='hist',
            device='cuda',
            scale_pos_weight=scale_pos_weight,
            random_state=seed,
            n_jobs=-1,
        )
    if model_name == 'lightgbm':
        if lgb is None:
            raise RuntimeError('LightGBM not available')
        return lgb.LGBMClassifier(
            n_estimators=500,
            learning_rate=0.04,
            num_leaves=63,
            class_weight='balanced',
            random_state=seed,
            n_jobs=-1,
            verbose=-1,
        )
    raise ValueError(model_name)

def get_scores(model, X):
    return model.predict_proba(X)[:, 1]

def choose_threshold(y_val, scores_val, c_fn=100, c_fp=1):
    best = None
    for tau in np.round(np.arange(0.01, 0.91, 0.01), 2):
        pred = (scores_val >= tau).astype(int)
        tn, fp, fn, tp = confusion_matrix(y_val, pred, labels=[0, 1]).ravel()
        cost = c_fp * fp + c_fn * fn
        mcc = matthews_corrcoef(y_val, pred) if len(np.unique(pred)) > 1 else 0.0
        candidate = (cost, fn, fp, -mcc, tau)
        if best is None or candidate < best:
            best = candidate
    return float(best[-1])

def evaluate_predictions(y_true, scores, tau):
    pred = (scores >= tau).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, pred, labels=[0, 1]).ravel()
    n = len(y_true)
    return {
        'threshold': float(tau),
        'n_test': int(n),
        'tp': int(tp),
        'fp': int(fp),
        'tn': int(tn),
        'fn': int(fn),
        'recall': float(recall_score(y_true, pred, zero_division=0)),
        'precision': float(precision_score(y_true, pred, zero_division=0)),
        'f1': float(f1_score(y_true, pred, zero_division=0)),
        'mcc': float(matthews_corrcoef(y_true, pred)) if len(np.unique(pred)) > 1 else 0.0,
        'average_precision': float(average_precision_score(y_true, scores)),
        'roc_auc': float(roc_auc_score(y_true, scores)) if len(np.unique(y_true)) > 1 else np.nan,
        'brier': float(brier_score_loss(y_true, scores)),
        'false_alerts_per_10k': float(fp / n * 10_000),
        'missed_frauds_per_10k': float(fn / n * 10_000),
        'cost_per_10k': float((fp + 100 * fn) / n * 10_000),
    }

In [ ]:
def split_indices(X, y, seed):
    idx = np.arange(len(y))
    train_val_idx, test_idx = train_test_split(idx, test_size=TEST_SIZE, stratify=y, random_state=seed)
    y_train_val = y.iloc[train_val_idx]
    train_idx, val_idx = train_test_split(train_val_idx, test_size=VALIDATION_SIZE, stratify=y_train_val, random_state=seed)
    return train_idx, val_idx, test_idx

def run_safe_pipeline(X, y, model_name, seed):
    train_idx, val_idx, test_idx = split_indices(X, y, seed)
    X_train, y_train = X.iloc[train_idx], y.iloc[train_idx]
    X_val, y_val = X.iloc[val_idx], y.iloc[val_idx]
    X_test, y_test = X.iloc[test_idx], y.iloc[test_idx]

    preprocessor = make_preprocessor(X_train)
    X_train_t = preprocessor.fit_transform(X_train)
    X_val_t = preprocessor.transform(X_val)
    X_test_t = preprocessor.transform(X_test)

    pos = max(int(np.sum(y_train == 1)), 1)
    neg = max(int(np.sum(y_train == 0)), 1)
    model = build_model(model_name, seed, scale_pos_weight=neg / pos)
    t0 = time.time()
    model.fit(X_train_t, y_train)
    fit_seconds = time.time() - t0

    val_scores = get_scores(model, X_val_t)
    tau = choose_threshold(y_val, val_scores)
    test_scores = get_scores(model, X_test_t)
    metrics = evaluate_predictions(y_test, test_scores, tau)
    metrics['fit_seconds'] = float(fit_seconds)
    return metrics

def run_unsafe_smote_before_split(X, y, model_name, seed):
    preprocessor = make_preprocessor(X)
    X_all_t = preprocessor.fit_transform(X)
    smote = SMOTE(k_neighbors=5, random_state=seed)
    X_res, y_res = smote.fit_resample(X_all_t, y)

    dummy = pd.DataFrame({'i': np.arange(len(y_res))})
    train_idx, val_idx, test_idx = split_indices(dummy, pd.Series(y_res), seed)
    X_train_t, y_train = X_res[train_idx], pd.Series(y_res).iloc[train_idx]
    X_val_t, y_val = X_res[val_idx], pd.Series(y_res).iloc[val_idx]
    X_test_t, y_test = X_res[test_idx], pd.Series(y_res).iloc[test_idx]

    pos = max(int(np.sum(y_train == 1)), 1)
    neg = max(int(np.sum(y_train == 0)), 1)
    model = build_model(model_name, seed, scale_pos_weight=neg / pos)
    t0 = time.time()
    model.fit(X_train_t, y_train)
    fit_seconds = time.time() - t0

    val_scores = get_scores(model, X_val_t)
    tau = choose_threshold(y_val, val_scores)
    test_scores = get_scores(model, X_test_t)
    metrics = evaluate_predictions(y_test, test_scores, tau)
    metrics['fit_seconds'] = float(fit_seconds)
    metrics['unsafe_resampled_rows'] = int(X_res.shape[0])
    return metrics

In [ ]:
def scenarios_for_dataset(dataset_name):
    if dataset_name in ['D1_creditcard', 'D2_fraudTest']:
        return ['SAFE_STRICT', 'UNSAFE_SMOTE_BEFORE_SPLIT']
    if dataset_name == 'D3_PaySim':
        return ['D3_STRICT_SAFE', 'D3_LEDGER_STATE_SAFE']
    raise ValueError(dataset_name)

def run_one(seed, dataset_name, scenario, model_name):
    if is_done(seed, dataset_name, scenario, model_name):
        append_log(f'SKIP seed={seed} {dataset_name} / {scenario} / {model_name}')
        return pd.read_csv(checkpoint_path(seed, dataset_name, scenario, model_name)).iloc[0].to_dict()

    df, label_col = load_dataset(dataset_name, seed)

    if scenario == 'UNSAFE_SMOTE_BEFORE_SPLIT':
        X, y = prepare_features(df, dataset_name, 'strict')
        metrics = run_unsafe_smote_before_split(X, y, model_name, seed)
        feature_policy = 'strict'
    elif scenario == 'SAFE_STRICT':
        X, y = prepare_features(df, dataset_name, 'strict')
        metrics = run_safe_pipeline(X, y, model_name, seed)
        feature_policy = 'strict'
    elif scenario == 'D3_STRICT_SAFE':
        X, y = prepare_features(df, dataset_name, 'strict')
        metrics = run_safe_pipeline(X, y, model_name, seed)
        feature_policy = 'strict'
    elif scenario == 'D3_LEDGER_STATE_SAFE':
        X, y = prepare_features(df, dataset_name, 'ledger_state')
        metrics = run_safe_pipeline(X, y, model_name, seed)
        feature_policy = 'ledger_state'
    else:
        raise ValueError(scenario)

    row = {
        'run_name': RUN_NAME,
        'timestamp': now(),
        'seed': seed,
        'dataset': dataset_name,
        'scenario': scenario,
        'model': model_name,
        'feature_policy': feature_policy,
        'smoke_test': SMOKE_TEST,
        'rows_used': int(len(df)),
        'fraud_rows_used': int((pd.to_numeric(df[label_col], errors='coerce') == 1).sum()),
        **metrics,
    }
    out = pd.DataFrame([row])
    write_csv_atomic(checkpoint_path(seed, dataset_name, scenario, model_name), out)
    append_log(f'DONE seed={seed} {dataset_name} / {scenario} / {model_name}: AP={row["average_precision"]:.4f}, MCC={row["mcc"]:.4f}')
    return row

all_rows = []
for seed in SEEDS:
    for dataset_name in ['D1_creditcard', 'D2_fraudTest', 'D3_PaySim']:
        for scenario in scenarios_for_dataset(dataset_name):
            for model_name in MODEL_NAMES:
                try:
                    row = run_one(seed, dataset_name, scenario, model_name)
                    all_rows.append(row)
                    write_csv_atomic(RESULTS_DIR / 'partial_results_latest.csv', pd.DataFrame(all_rows))
                except Exception as exc:
                    append_log(f'ERROR seed={seed} {dataset_name} / {scenario} / {model_name}: {repr(exc)}')
                    write_json(CHECKPOINT_DIR / f'ERROR__seed{seed}__{dataset_name}__{scenario}__{model_name}.json', {
                        'seed': seed,
                        'dataset': dataset_name,
                        'scenario': scenario,
                        'model': model_name,
                        'error': repr(exc),
                        'timestamp': now(),
                    })

frames = [pd.read_csv(p) for p in sorted(CHECKPOINT_DIR.glob('seed*.csv'))]
results = pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()
write_csv_atomic(RESULTS_DIR / 'leakage_stress_multiseed_results.csv', results)
results

In [ ]:
results = pd.read_csv(RESULTS_DIR / 'leakage_stress_multiseed_results.csv')

safe_map = {
    'D1_creditcard': 'SAFE_STRICT',
    'D2_fraudTest': 'SAFE_STRICT',
    'D3_PaySim': 'D3_STRICT_SAFE',
}

safe = results[results.apply(lambda r: r['scenario'] == safe_map[r['dataset']], axis=1)].copy()
safe = safe[['seed', 'dataset', 'model', 'average_precision', 'mcc', 'recall', 'precision', 'cost_per_10k', 'brier']]
safe = safe.rename(columns={
    'average_precision': 'safe_average_precision',
    'mcc': 'safe_mcc',
    'recall': 'safe_recall',
    'precision': 'safe_precision',
    'cost_per_10k': 'safe_cost_per_10k',
    'brier': 'safe_brier',
})

inflation = results.merge(safe, on=['seed', 'dataset', 'model'], how='left')
inflation['lii_average_precision'] = inflation['average_precision'] - inflation['safe_average_precision']
inflation['lii_mcc'] = inflation['mcc'] - inflation['safe_mcc']
inflation['lii_recall'] = inflation['recall'] - inflation['safe_recall']
inflation['lii_precision'] = inflation['precision'] - inflation['safe_precision']
inflation['cost_delta_per_10k'] = inflation['cost_per_10k'] - inflation['safe_cost_per_10k']
inflation['brier_delta'] = inflation['brier'] - inflation['safe_brier']

write_csv_atomic(RESULTS_DIR / 'leakage_inflation_multiseed_index.csv', inflation)

summary = (
    inflation[~inflation.apply(lambda r: r['scenario'] == safe_map[r['dataset']], axis=1)]
    .groupby(['dataset', 'scenario', 'model'])
    .agg(
        seeds=('seed', 'nunique'),
        ap_inflation_mean=('lii_average_precision', 'mean'),
        ap_inflation_std=('lii_average_precision', 'std'),
        mcc_inflation_mean=('lii_mcc', 'mean'),
        mcc_inflation_std=('lii_mcc', 'std'),
        cost_delta_mean=('cost_delta_per_10k', 'mean'),
        cost_delta_std=('cost_delta_per_10k', 'std'),
    )
    .reset_index()
)
write_csv_atomic(RESULTS_DIR / 'leakage_inflation_multiseed_summary.csv', summary)
summary

In [ ]:
inflation = pd.read_csv(RESULTS_DIR / 'leakage_inflation_multiseed_index.csv')
safe_mask = inflation.apply(lambda r: r['scenario'] == {'D1_creditcard':'SAFE_STRICT','D2_fraudTest':'SAFE_STRICT','D3_PaySim':'D3_STRICT_SAFE'}[r['dataset']], axis=1)
plot_df = inflation[~safe_mask].copy()

scenario_labels = {
    'UNSAFE_SMOTE_BEFORE_SPLIT': 'SMOTE before split',
    'D3_LEDGER_STATE_SAFE': 'Ledger-state features',
}
dataset_labels = {'D1_creditcard': 'D1', 'D2_fraudTest': 'D2', 'D3_PaySim': 'D3'}
model_labels = {'xgboost': 'XGBoost', 'lightgbm': 'LightGBM'}

plot_df['scenario_label'] = plot_df['scenario'].map(scenario_labels)
plot_df['dataset_label'] = plot_df['dataset'].map(dataset_labels)
plot_df['model_label'] = plot_df['model'].map(model_labels)

for metric, ylabel in [
    ('lii_average_precision', 'Mean AP inflation'),
    ('lii_mcc', 'Mean MCC inflation'),
    ('cost_delta_per_10k', 'Mean cost delta per 10,000'),
]:
    fig, ax = plt.subplots(figsize=(7.0, 4.0))
    sns.barplot(
        data=plot_df,
        x='dataset_label',
        y=metric,
        hue='model_label',
        errorbar='sd',
        edgecolor='black',
        linewidth=0.45,
        ax=ax,
    )
    ax.axhline(0, color='black', linewidth=0.8)
    ax.set_xlabel('Dataset')
    ax.set_ylabel(ylabel)
    ax.set_title(ylabel + ' across five seeds')
    ax.legend(title='Model', loc='best', frameon=True)
    ax.grid(axis='y', linestyle='--', linewidth=0.45, alpha=0.55)
    ax.grid(axis='x', visible=False)
    for spine in ['top', 'right']:
        ax.spines[spine].set_visible(False)
    fig.tight_layout()
    png = FIGURES_DIR / f'multiseed_{metric}_by_dataset.png'
    pdf = FIGURES_DIR / f'multiseed_{metric}_by_dataset.pdf'
    fig.savefig(png, dpi=600, bbox_inches='tight', facecolor='white')
    fig.savefig(pdf, bbox_inches='tight', facecolor='white')
    plt.show()
    append_log(f'Saved figure: {png}')

summary = pd.read_csv(RESULTS_DIR / 'leakage_inflation_multiseed_summary.csv')
summary['row_label'] = summary['dataset'].map(dataset_labels) + ' - ' + summary['model'].map(model_labels)
summary['scenario_label'] = summary['scenario'].map(scenario_labels)
heat = summary.pivot_table(index='row_label', columns='scenario_label', values='ap_inflation_mean', aggfunc='mean')

fig, ax = plt.subplots(figsize=(6.8, max(3.0, 0.45 * len(heat))))
sns.heatmap(
    heat,
    annot=True,
    fmt='.3f',
    cmap='vlag',
    center=0,
    linewidths=0.5,
    linecolor='white',
    cbar_kws={'label': 'Mean AP inflation'},
    annot_kws={'fontsize': 8},
    ax=ax,
)
ax.set_title('Mean AP Leakage Inflation across Five Seeds')
ax.set_xlabel('Stress scenario')
ax.set_ylabel('Dataset and model')
ax.tick_params(axis='x', rotation=0)
ax.tick_params(axis='y', rotation=0)
fig.tight_layout()
png = FIGURES_DIR / 'multiseed_ap_leakage_inflation_heatmap.png'
pdf = FIGURES_DIR / 'multiseed_ap_leakage_inflation_heatmap.pdf'
fig.savefig(png, dpi=600, bbox_inches='tight', facecolor='white')
fig.savefig(pdf, bbox_inches='tight', facecolor='white')
plt.show()
append_log(f'Saved figure: {png}')

In [ ]:
summary = {
    'run_name': RUN_NAME,
    'completed_at': now(),
    'project_root': str(PROJECT_ROOT),
    'experiment_root': str(EXPERIMENT_ROOT),
    'seeds': SEEDS,
    'results_file': str(RESULTS_DIR / 'leakage_stress_multiseed_results.csv'),
    'inflation_file': str(RESULTS_DIR / 'leakage_inflation_multiseed_index.csv'),
    'summary_file': str(RESULTS_DIR / 'leakage_inflation_multiseed_summary.csv'),
    'figures_dir': str(FIGURES_DIR),
    'checkpoint_dir': str(CHECKPOINT_DIR),
    'n_result_rows': int(len(pd.read_csv(RESULTS_DIR / 'leakage_stress_multiseed_results.csv'))),
}
write_json(EXPERIMENT_ROOT / 'run_summary.json', summary)
summary

## Expected Runtime on Colab T4

Expected runtime is approximately **45-75 minutes** for the full five-seed run.

Why:

- The first one-seed notebook completed 24 result rows in about 24 minutes.
- This notebook runs 60 result rows, but it removes logistic regression and preprocessing-only stress, so it is not 2.5x slower in practice.
- D1/D2 unsafe SMOTE-before-split jobs are the heaviest jobs.
- Google Drive read/write speed can add variability.

If it disconnects, rerun from the top. Existing checkpoint files are skipped.